In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from datetime import datetime, timedelta

print('all good')

In [ ]:
np.random.seed(42)

start = datetime(2024, 1, 1)
n = 60 * 24
timestamps = [start + timedelta(hours=i) for i in range(n)]

hours = np.array([t.hour for t in timestamps])
days  = np.array([i // 24 for i in range(n)])

temp = 25 + 7 * np.sin(2 * np.pi * (hours - 6) / 24) + 0.03 * days + np.random.normal(0, 1.0, n)

hum = np.clip(70 - 0.8 * (temp - 25) + np.random.normal(0, 3, n), 20, 100)

aqi = np.clip(
    50 + 20 * np.exp(-((hours - 8)**2) / 4) + 15 * np.exp(-((hours - 17)**2) / 4) + np.random.normal(0, 5, n),
    0, 200
)


co2 = np.clip(
    400 + 50 * np.exp(-((hours - 9)**2) / 6) + 40 * np.exp(-((hours - 18)**2) / 6) + np.random.normal(0, 10, n),
    380, 700
)

df = pd.DataFrame({
    'timestamp': timestamps,
    'temperature': np.round(temp, 2),
    'humidity': np.round(hum, 2),
    'aqi': np.round(aqi, 2),
    'co2_ppm': np.round(co2, 2)
})

for col in ['temperature', 'humidity', 'aqi', 'co2_ppm']:
    missing_idx = np.random.choice(df.index, size=int(0.03 * len(df)), replace=False)
    df.loc[missing_idx, col] = np.nan

print('dataset shape:', df.shape)
df.head()

In [ ]:
print('missing values:')
print(df.isnull().sum())

In [ ]:

df.sort_values('timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True) 

print('after filling:')
print(df.isnull().sum())

In [ ]:
sensor_cols = ['temperature', 'humidity', 'aqi', 'co2_ppm']

for col in sensor_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    print(f'{col}: clipped to [{lower:.2f}, {upper:.2f}]')

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['day_of_year'] = df['timestamp'].dt.dayofyear
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)


df['temp_lag1'] = df['temperature'].shift(1)
df['temp_lag2'] = df['temperature'].shift(2)
df['temp_lag24'] = df['temperature'].shift(24)

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print('final shape after feature engineering:', df.shape)
df.describe().round(2)

In [ ]:
week_data = df[df['day_of_year'] <= df['day_of_year'].min() + 6]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Sensor Readings - First 7 Days', fontsize=14)

axes[0].plot(week_data['timestamp'], week_data['temperature'], color='tomato')
axes[0].set_ylabel('Temperature (C)')
axes[0].set_title('Temperature')

axes[1].plot(week_data['timestamp'], week_data['humidity'], color='steelblue')
axes[1].set_ylabel('Humidity (%)')
axes[1].set_title('Humidity')

axes[2].plot(week_data['timestamp'], week_data['aqi'], color='darkorange')
axes[2].set_ylabel('AQI')
axes[2].set_title('Air Quality Index')

axes[3].plot(week_data['timestamp'], week_data['co2_ppm'], color='green')
axes[3].set_ylabel('CO2 (ppm)')
axes[3].set_title('CO2 Level')

plt.tight_layout()
plt.show()

In [ ]:
hourly = df.groupby('hour')[sensor_cols].mean()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Average Sensor Values by Hour', fontsize=13)

colors_list = ['tomato', 'steelblue', 'darkorange', 'green']
for ax, col, c in zip(axes.flat, sensor_cols, colors_list):
    ax.plot(hourly.index, hourly[col], color=c, linewidth=2, marker='o', markersize=3)
    ax.set_title(col)
    ax.set_xlabel('Hour of Day')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
corr_matrix = df[sensor_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation between Sensors')
plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Distribution of Sensor Values', fontsize=13)

for ax, col, c in zip(axes.flat, sensor_cols, colors_list):
    ax.hist(df[col], bins=40, color=c, edgecolor='white', alpha=0.8)
    ax.axvline(df[col].mean(), color='black', linestyle='--', label=f'mean = {df[col].mean():.1f}')
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
features = ['hour', 'day_of_week', 'day_of_year', 'month', 'is_weekend',
            'humidity', 'aqi', 'co2_ppm', 'temp_lag1', 'temp_lag2', 'temp_lag24']

X = df[features]
y = df['temperature']


split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print('training samples:', len(X_train))
print('testing samples:', len(X_test))

In [ ]:

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest trained!')

In [ ]:
scaler = StandardScaler()
lr = LinearRegression()
lr.fit(scaler.fit_transform(X_train), y_train)
y_pred_lr = lr.predict(scaler.transform(X_test))

print('Linear Regression trained!')

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'--- {name} ---')
    print(f'  MAE  = {mae:.4f}')
    print(f'  RMSE = {rmse:.4f}')
    print(f'  R2   = {r2:.4f}')
    print()
    return mae, rmse, r2

rf_scores = evaluate_model('Random Forest', y_test, y_pred_rf)
lr_scores = evaluate_model('Linear Regression', y_test, y_pred_lr)



In [ ]:
test_timestamps = df['timestamp'].iloc[split:].reset_index(drop=True)

plt.figure(figsize=(14, 5))
plt.plot(test_timestamps[:200], y_test.values[:200], label='Actual', color='steelblue', linewidth=2)
plt.plot(test_timestamps[:200], y_pred_rf[:200], label='Predicted (RF)', color='tomato', linestyle='--', linewidth=1.8)
plt.title('Actual vs Predicted Temperature')
plt.xlabel('Date')
plt.ylabel('Temperature (C)')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot.barh(color='steelblue', edgecolor='white')
plt.title('Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

